# 🌾 KrishiRakshak — YOLOv8 Plant Disease Training

This notebook trains a YOLOv8 classification model on the **PlantVillage dataset** (38 disease classes, ~87,000 images).

**Steps:**
1. Connect to GPU runtime
2. Install dependencies
3. Download PlantVillage dataset from Kaggle
4. Train YOLOv8
5. Download the trained model (`best.pt`)
6. Place it in your local `yolo_weights/` folder

**Expected time: ~20 minutes on T4 GPU**

---
### Before running:
- Go to **Runtime → Change runtime type → T4 GPU** → Save
- Get your Kaggle API token from https://www.kaggle.com/settings/api


In [ ]:
# ── Cell 1: Check GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install ultralytics kaggle --quiet
import ultralytics
print(f'Ultralytics version: {ultralytics.__version__}')

In [ ]:
# ── Cell 3: Kaggle credentials ───────────────────────────────────────────────
# Option A: Upload kaggle.json file
from google.colab import files
import os, json

print('Upload your kaggle.json file (from https://www.kaggle.com/settings/api)')
uploaded = files.upload()   # click 'Choose Files' and select kaggle.json

# Save to ~/.kaggle/
os.makedirs('/root/.kaggle', exist_ok=True)
for fname, content in uploaded.items():
    with open(f'/root/.kaggle/{fname}', 'wb') as f:
        f.write(content)
!chmod 600 /root/.kaggle/kaggle.json
print('Kaggle credentials saved!')

In [ ]:
# ── Cell 3B: Alternative — paste token directly (if file upload doesn't work) ─
# UNCOMMENT this cell and fill in your username and key from kaggle.json
# import os, json
# os.makedirs('/root/.kaggle', exist_ok=True)
# creds = {"username": "YOUR_KAGGLE_USERNAME", "key": "YOUR_KAGGLE_KEY"}
# with open('/root/.kaggle/kaggle.json', 'w') as f:
#     json.dump(creds, f)
# !chmod 600 /root/.kaggle/kaggle.json
# print('Credentials set!')

In [ ]:
# ── Cell 4: Download PlantVillage dataset ────────────────────────────────────
import os
os.makedirs('/content/data', exist_ok=True)

print('Downloading PlantVillage dataset (~2.7 GB) ...')
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content/data --unzip
print('Download complete!')

In [ ]:
# ── Cell 5: Find dataset structure ──────────────────────────────────────────
from pathlib import Path

def find_train_valid(search_root):
    for train_dir in sorted(Path(search_root).rglob('train')):
        if not train_dir.is_dir():
            continue
        subdirs = [d for d in train_dir.iterdir() if d.is_dir()]
        if len(subdirs) < 2:
            continue
        for vname in ('valid', 'validation', 'val'):
            valid_dir = train_dir.parent / vname
            if valid_dir.exists():
                return train_dir, valid_dir
        return train_dir, None
    return None, None

train_dir, valid_dir = find_train_valid('/content/data')

if train_dir is None:
    print('ERROR: Could not find train/ directory. Check that download succeeded.')
    # List what we have
    !find /content/data -type d | head -20
else:
    data_root = train_dir.parent
    classes = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
    print(f'Dataset root : {data_root}')
    print(f'Train dir    : {train_dir}')
    print(f'Valid dir    : {valid_dir}')
    print(f'Classes      : {len(classes)}')
    print(f'First 5      : {classes[:5]}')

In [ ]:
# ── Cell 6: Train YOLOv8 ────────────────────────────────────────────────────
from ultralytics import YOLO
import torch

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Load YOLOv8 nano classification model (fastest, good accuracy)
model = YOLO('yolov8n-cls.pt')

# Train on PlantVillage
results = model.train(
    data=str(data_root),
    epochs=30,            # 30 epochs ~= 20 min on T4 GPU
    imgsz=224,
    batch=64,
    device='0',           # GPU
    project='/content/runs/classify',
    name='plant_disease',
    exist_ok=True,
    patience=7,           # early stopping
    save=True,
    plots=True,
    verbose=True,
    workers=4,
)

print('Training complete!')

In [ ]:
# ── Cell 7: Evaluate and show results ───────────────────────────────────────
from pathlib import Path
from IPython.display import Image, display
import glob

run_dir = Path('/content/runs/classify/plant_disease')
best_pt = run_dir / 'weights' / 'best.pt'

print(f'Best model: {best_pt}')
print(f'Size: {best_pt.stat().st_size / 1024 / 1024:.1f} MB')

# Show training plots
for plot in ['results.png', 'confusion_matrix.png']:
    p = run_dir / plot
    if p.exists():
        print(f'\n{plot}:')
        display(Image(str(p), width=800))

# Quick validation
print('\nRunning validation...')
val_model = YOLO(str(best_pt))
metrics = val_model.val(data=str(data_root), imgsz=224, batch=64, device='0')
print(f'Top-1 Accuracy: {metrics.top1:.3f}')
print(f'Top-5 Accuracy: {metrics.top5:.3f}')

In [ ]:
# ── Cell 8: Test on a sample image ──────────────────────────────────────────
import glob, random
from pathlib import Path
from IPython.display import Image as IPImage, display
from ultralytics import YOLO

# Pick a random validation image
val_images = glob.glob(str(valid_dir / '*' / '*.jpg'))
if not val_images:
    val_images = glob.glob(str(valid_dir / '*' / '*.JPG'))
sample = random.choice(val_images)
true_label = Path(sample).parent.name

print(f'Test image : {Path(sample).name}')
print(f'True label : {true_label}')

# Run inference
best_pt = Path('/content/runs/classify/plant_disease/weights/best.pt')
model = YOLO(str(best_pt))
results = model(sample, verbose=False)

names = results[0].names
probs = results[0].probs
top5_idx = probs.top5
top5_conf = probs.top5conf.tolist()

print('\nTop-5 predictions:')
for idx, conf in zip(top5_idx, top5_conf):
    marker = ' <-- CORRECT' if names[idx] == true_label else ''
    print(f'  {names[idx]}: {conf*100:.1f}%{marker}')

display(IPImage(sample, width=300))

In [ ]:
# ── Cell 9: Download the model to your computer ──────────────────────────────
from google.colab import files
from pathlib import Path
import shutil

best_pt = Path('/content/runs/classify/plant_disease/weights/best.pt')

# Copy with a descriptive name
dest = Path('/content/plant_disease_krishirakshak.pt')
shutil.copy(best_pt, dest)

print(f'Downloading: {dest.name}  ({dest.stat().st_size/1024/1024:.1f} MB)')
print('The file will appear in your Downloads folder.')
print()
print('AFTER DOWNLOAD — do this on your local machine:')
print('  1. Move the file to:  KrishiRakshak/yolo_weights/plant_disease_krishirakshak.pt')
print('  2. In your .env file set:')
print('       YOLO_MODEL_PATH=yolo_weights/plant_disease_krishirakshak.pt')
print('  3. Restart backend:  uvicorn main:app --reload')

files.download(str(dest))

## After downloading `plant_disease_krishirakshak.pt`

Run these commands on your **local machine** (KrishiRakshak folder):

```powershell
# 1. Move the model file
Move-Item "$env:USERPROFILE\Downloads\plant_disease_krishirakshak.pt" .\yolo_weights\

# 2. Update .env  
(Get-Content .env) -replace 'YOLO_MODEL_PATH=.*', 'YOLO_MODEL_PATH=yolo_weights/plant_disease_krishirakshak.pt' | Set-Content .env

# 3. Verify it works
python scripts/smoke_test_yolo.py

# 4. Start the backend
cd backend
python -m uvicorn main:app --reload --port 8000
```

Then open **http://localhost:3000** → click **Disease Detection** tab → upload a leaf photo!
